# CFB Extended Factor Analysis

How do these factors affect spread/O-U accuracy?
- Weather (temperature, wind, rain, snow, dome)
- Coaching (first-year coach, record vs spread)
- Returning production (NIL/transfer portal net effect)
- Recruiting talent gap
- NFL draft pipeline
- Rivalry games
- Travel distance

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import math

ROOT = Path().resolve().parent
DB = duckdb.connect(str(ROOT / 'data/warehouse/ons.duckdb'), read_only=True)
plt.style.use('dark_background')

print('Tables:')
DB.execute("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema = 'cfbd' ORDER BY table_name").df()

## Extended edge factors overview

In [ ]:
edges = DB.execute("""
    SELECT *,
           ats_pct - 50 as ats_edge,
           over_pct - 50 as ou_edge
    FROM main_marts.mart_cfbd_extended_edges
    WHERE games >= 30
    ORDER BY abs(ats_pct - 50) DESC
""").df()

print(f'Total factor combinations: {len(edges)}')
display(edges)

## Weather analysis

In [ ]:
weather = DB.execute("""
    SELECT
        temp_bucket,
        high_wind,
        rain_game,
        snow_game,
        game_indoors,
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end)*100,1) as ats_pct,
        round(avg(case when ou_result='over' then 1.0 else 0.0 end)*100,1) as over_pct,
        round(avg(total_vs_ou),1) as avg_total_vs_ou
    FROM main_marts.mart_cfbd_game_context
    WHERE spread_result in ('covered','missed')
    GROUP BY 1,2,3,4,5
    HAVING count(*) >= 20
    ORDER BY over_pct
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wind vs O/U
wind_df = weather.groupby('high_wind').agg({'games':'sum','over_pct':'mean','ats_pct':'mean'}).reset_index()
axes[0].bar(['Normal Wind','High Wind (20+mph)'], wind_df['over_pct'],
            color=['steelblue','orange'])
axes[0].axhline(50, color='white', linestyle='--', alpha=0.5)
axes[0].set_title('Over % by Wind Speed')
axes[0].set_ylabel('Over %')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# Temp bucket vs O/U
temp_df = weather.groupby('temp_bucket').agg({'over_pct':'mean','games':'sum'}).reset_index()
axes[1].bar(temp_df['temp_bucket'], temp_df['over_pct'],
            color=['steelblue' if v > 50 else 'orange' for v in temp_df['over_pct']])
axes[1].axhline(50, color='white', linestyle='--', alpha=0.5)
axes[1].set_title('Over % by Temperature')
axes[1].tick_params(axis='x', rotation=30)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## Returning production (NIL / transfer portal effect)

In [ ]:
returning = DB.execute("""
    SELECT
        returning_edge,
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end)*100,1) as ats_pct,
        round(avg(case when ou_result='over' then 1.0 else 0.0 end)*100,1) as over_pct
    FROM main_marts.mart_cfbd_game_context
    WHERE spread_result in ('covered','missed') AND returning_edge IS NOT NULL
    GROUP BY returning_edge
    ORDER BY ats_pct DESC
""").df()

fig, ax = plt.subplots(figsize=(10,5))
colors = ['#2ecc71' if v > 50 else '#e74c3c' for v in returning['ats_pct']]
ax.barh(returning['returning_edge'], returning['ats_pct'] - 50, color=colors)
ax.axvline(0, color='white', linestyle='--', alpha=0.5)
ax.set_xlabel('ATS edge vs 50%')
ax.set_title('ATS Cover Rate by Returning Production Advantage')
plt.tight_layout()
plt.show()
display(returning)

## Rivalry games

In [ ]:
rivalry = DB.execute("""
    SELECT
        is_rivalry_game,
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end)*100,1) as ats_pct,
        round(avg(case when ou_result='over' then 1.0 else 0.0 end)*100,1) as over_pct,
        round(avg(spread_abs),1) as avg_spread,
        round(avg(abs(margin_vs_spread)),1) as avg_margin_error
    FROM main_marts.mart_cfbd_game_context
    WHERE spread_result in ('covered','missed')
    GROUP BY is_rivalry_game
""").df()

print('Rivalry vs Non-Rivalry:')
display(rivalry)

## Travel distance effect

In [ ]:
# Travel distance requires joining game venues — compute haversine distance
# We need to join games → venues for away team's home venue vs game venue
travel = DB.execute("""
    SELECT
        g.away_team,
        g.home_team,
        g.season,
        g.week,
        -- We approximate: if away team conference != home team conference, likely long travel
        -- Full haversine would need team home venue lookup
        g.home_conference != g.away_conference as cross_conference_travel,
        la.ats_pct,
        la.spread_covered
    FROM cfbd.games g
    JOIN main_marts.mart_cfbd_line_accuracy la 
        ON g.game_id = la.game_id
    WHERE la.spread_result in ('covered','missed')
      AND g.home_conference IS NOT NULL
      AND g.away_conference IS NOT NULL
""").df()

cross_conf = travel.groupby('cross_conference_travel').agg(
    games=('spread_covered','count'),
    ats_pct=('spread_covered', lambda x: round(x.mean()*100, 1))
).reset_index()
cross_conf['label'] = cross_conf['cross_conference_travel'].map({True: 'Cross-conference travel', False: 'Same conference'})
print('Cover rate by travel type:')
display(cross_conf[['label','games','ats_pct']])

## Coach analysis — ATS record by coach

In [ ]:
coach_ats = DB.execute("""
    SELECT
        home_coach as coach,
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end)*100,1) as ats_pct,
        round(avg(case when ou_result='over' then 1.0 else 0.0 end)*100,1) as over_pct
    FROM main_marts.mart_cfbd_game_context
    WHERE spread_result in ('covered','missed')
      AND home_coach IS NOT NULL
    GROUP BY home_coach
    HAVING count(*) >= 15
    ORDER BY ats_pct DESC
    LIMIT 20
""").df()

print('Top 20 coaches ATS as home team:')
display(coach_ats)

## Recruiting talent gap

In [ ]:
talent = DB.execute("""
    SELECT
        talent_gap_bucket,
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end)*100,1) as ats_pct,
        round(avg(case when ou_result='over' then 1.0 else 0.0 end)*100,1) as over_pct
    FROM main_marts.mart_cfbd_game_context
    WHERE spread_result in ('covered','missed')
      AND talent_gap_bucket IS NOT NULL
    GROUP BY talent_gap_bucket
    ORDER BY ats_pct DESC
""").df()

print('ATS cover rate by recruiting talent gap:')
display(talent)